## Managing Global Scope Variables in Python

In Python, variables declared outside of any function or class are considered global variables. They can be accessed from anywhere in the script. However, to *modify* a global variable from within a function, you need to explicitly declare it using the `global` keyword. Otherwise, Python will treat it as a new local variable.

In [1]:
# Declare a global variable
global_message = "Hello from the global scope!"
global_counter = 0

def access_global():
    # This function can access the global_message variable
    print(f"Inside access_global: {global_message}")

def modify_global():
    # To modify a global variable, you must use the 'global' keyword
    global global_counter
    global_counter += 1
    print(f"Inside modify_global: global_counter is now {global_counter}")

def try_to_modify_without_global():
    # This will create a NEW local variable named global_message_local
    # and will NOT affect the global_message variable.
    global_message_local = "This is a local variable."
    print(f"Inside try_to_modify_without_global (local): {global_message_local}")


# --- Demonstrate Usage ---
print(f"Initial global_message: {global_message}")
print(f"Initial global_counter: {global_counter}")

access_global()

modify_global() # First call
modify_global() # Second call

print(f"Global counter after modifications: {global_counter}")

try_to_modify_without_global()
print(f"Global message after function call (unaffected): {global_message}")

# Demonstrate trying to access the local variable outside its scope
try:
    print(global_message_local)
except NameError as e:
    print(f"\nError trying to access local_message outside its scope: {e}")

Initial global_message: Hello from the global scope!
Initial global_counter: 0
Inside access_global: Hello from the global scope!
Inside modify_global: global_counter is now 1
Inside modify_global: global_counter is now 2
Global counter after modifications: 2
Inside try_to_modify_without_global (local): This is a local variable.
Global message after function call (unaffected): Hello from the global scope!

Error trying to access local_message outside its scope: name 'global_message_local' is not defined


### Explanation:

1.  **`global_message = "Hello from the global scope!"`**: This line declares `global_message` as a global variable.

2.  **`access_global()` function**: This function can *read* the `global_message` without any special keywords. Python's scope resolution (LEGB rule: Local -> Enclosing function locals -> Global -> Built-in) allows functions to look up variables in the global scope.

3.  **`modify_global()` function**: To *change* the value of `global_counter`, we use `global global_counter`. This tells Python that `global_counter` inside this function refers to the global variable, not a new local one.

4.  **`try_to_modify_without_global()` function**: If you try to assign a value to a variable that has the same name as a global variable but without using the `global` keyword (like `global_message_local = "This is a local variable." `), Python creates a *new local variable* with that name, leaving the global variable untouched. The `NameError` demonstrates that this local variable is only accessible within the function where it was created.

## Python Namespaces Example

A namespace is a mapping from names to objects. Different namespaces can coexist independently. Python has several types of namespaces:

*   **Built-in namespace**: Contains all the built-in functions and exceptions.
*   **Global namespace**: Contains names defined at the module level (outside any function or class).
*   **Local namespace**: Created when a function is called, containing names defined within that function. Each function call creates a new local namespace.

Python follows the **LEGB rule** (Local -> Enclosing function locals -> Global -> Built-in) to resolve names when they are referenced.

In [2]:
# 1. Global Namespace
global_var = "I'm a global variable!"

def outer_function():
    # 2. Enclosing Function's Local Namespace
    enclosing_var = "I'm from the enclosing function!"

    def inner_function():
        # 3. Local Namespace of inner_function
        local_var = "I'm a local variable in inner_function!"

        print(f"\nInside inner_function:")
        print(f"  Accessing local_var: {local_var}")
        print(f"  Accessing enclosing_var: {enclosing_var}") # Accessing from enclosing scope
        print(f"  Accessing global_var: {global_var}")       # Accessing from global scope
        # Accessing a built-in function
        print(f"  Accessing built-in len(): {len('hello')}")

        # Trying to modify global_var without 'global' keyword
        # This would create a new local variable if uncommented
        # global_var = "I'm a local copy of global_var"

    inner_function()
    print(f"\nInside outer_function (after inner_function call):")
    print(f"  Accessing enclosing_var: {enclosing_var}")
    print(f"  Accessing global_var: {global_var}")
    try:
        # Cannot access local_var from inner_function here
        print(local_var)
    except NameError as e:
        print(f"  Attempted to access local_var (expected NameError): {e}")

# --- Program Execution ---
print(f"At the global level:")
print(f"  Accessing global_var: {global_var}")

outer_function()

print(f"\nAt the global level (after function calls):")
print(f"  Accessing global_var: {global_var}")
try:
    # Cannot access enclosing_var here
    print(enclosing_var)
except NameError as e:
    print(f"  Attempted to access enclosing_var (expected NameError): {e}")

At the global level:
  Accessing global_var: I'm a global variable!

Inside inner_function:
  Accessing local_var: I'm a local variable in inner_function!
  Accessing enclosing_var: I'm from the enclosing function!
  Accessing global_var: I'm a global variable!
  Accessing built-in len(): 5

Inside outer_function (after inner_function call):
  Accessing enclosing_var: I'm from the enclosing function!
  Accessing global_var: I'm a global variable!
  Attempted to access local_var (expected NameError): name 'local_var' is not defined

At the global level (after function calls):
  Accessing global_var: I'm a global variable!
  Attempted to access enclosing_var (expected NameError): name 'enclosing_var' is not defined


### Observations from the output:

*   **Global Scope**: `global_var` is accessible from anywhere.
*   **Enclosing Scope**: `enclosing_var` is accessible within `outer_function` and `inner_function` (because `inner_function` is nested inside `outer_function`).
*   **Local Scope**: `local_var` is only accessible within `inner_function`. Trying to access it outside of `inner_function` (even from its enclosing `outer_function`) results in a `NameError`.
*   **LEGB Rule**: Python successfully finds variables by first checking the `L`ocal scope, then `E`nclosing function scope, then `G`lobal scope, and finally the `B`uilt-in scope.

This example illustrates how Python manages variable visibility and lifetime through its namespace system.

## Understanding Block Scope in Python (or the lack thereof)

In many programming languages (like C++, Java, JavaScript with `let`/`const`), a "block scope" is created by constructs like `if` statements or `for` loops, meaning variables declared inside these blocks are only accessible within that block. Python does not create new scopes for `if`/`else` blocks, `for` loops, or `while` loops. Variables defined within these constructs belong to the enclosing function's scope or the global scope.

In [3]:
# Global scope variable
x = 10

def function_scope_example():
    # Function scope variable
    y = 20

    print(f"\nInside function_scope_example:")
    print(f"  Accessing global x: {x}")
    print(f"  Accessing local y: {y}")

    # --- Demonstrating 'if' block behavior ---
    if True:
        # 'z' is defined inside the 'if' block, but it's part of the function's scope
        z = 30
        print(f"  Inside 'if' block: z = {z}")

    # 'z' is still accessible outside the 'if' block within the same function
    print(f"  Outside 'if' block, but inside function: z = {z}")

    # --- Demonstrating 'for' loop behavior ---
    print(f"\n  Demonstrating 'for' loop scope:")
    for i in range(3):
        loop_var = f"Iteration {i}"
        print(f"    Inside 'for' loop: loop_var = {loop_var}")

    # 'loop_var' is still accessible outside the 'for' loop within the same function
    # It retains the value from the last iteration
    print(f"  Outside 'for' loop, but inside function: loop_var (last value) = {loop_var}")

    # --- Demonstrating 'while' loop behavior ---
    print(f"\n  Demonstrating 'while' loop scope:")
    count = 0
    while count < 2:
        while_var = f"While count {count}"
        print(f"    Inside 'while' loop: while_var = {while_var}")
        count += 1

    # 'while_var' is still accessible outside the 'while' loop within the same function
    # It retains the value from the last iteration
    print(f"  Outside 'while' loop, but inside function: while_var (last value) = {while_var}")


# --- Global level execution ---
print(f"At the global level: x = {x}")

function_scope_example()

print(f"\nBack at the global level:")
print(f"  Accessing global x: {x}")

try:
    # Trying to access y (function scope) from global scope - this will fail
    print(y)
except NameError as e:
    print(f"  Attempted to access y (expected NameError): {e}")

try:
    # Trying to access z (from if block) from global scope - this will fail
    print(z)
except NameError as e:
    print(f"  Attempted to access z (expected NameError): {e}")

At the global level: x = 10

Inside function_scope_example:
  Accessing global x: 10
  Accessing local y: 20
  Inside 'if' block: z = 30
  Outside 'if' block, but inside function: z = 30

  Demonstrating 'for' loop scope:
    Inside 'for' loop: loop_var = Iteration 0
    Inside 'for' loop: loop_var = Iteration 1
    Inside 'for' loop: loop_var = Iteration 2
  Outside 'for' loop, but inside function: loop_var (last value) = Iteration 2

  Demonstrating 'while' loop scope:
    Inside 'while' loop: while_var = While count 0
    Inside 'while' loop: while_var = While count 1
  Outside 'while' loop, but inside function: while_var (last value) = While count 1

Back at the global level:
  Accessing global x: 10
  Attempted to access y (expected NameError): name 'y' is not defined
  Attempted to access z (expected NameError): name 'z' is not defined


### Observations on Block Scope in Python:

*   **Global Variables (`x`)**: Accessible everywhere.
*   **Function-Local Variables (`y`)**: Accessible throughout the `function_scope_example` function, but not outside it.
*   **`if` Block Variables (`z`)**: Even though `z` was defined inside an `if` statement, it was accessible *outside* that `if` block, but still within the `function_scope_example`.
*   **`for` Loop Variables (`loop_var`, `i`)**: Similarly, `loop_var` and the loop counter `i` were accessible *outside* the `for` loop, retaining their last assigned values.
*   **`while` Loop Variables (`while_var`, `count`)**: Variables defined inside a `while` loop are also accessible after the loop finishes.

This behavior highlights that Python primarily uses **function scope** (and module/global scope), not block scope for conditional statements or loops. The only constructs that create new scopes for variables in Python are functions, classes, and modules themselves.